In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

#### import the dataset


In [3]:
df = pd.read_csv("../Data/playstore/reviews.csv")

In [4]:
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,sortOrder,appId
0,gp:AOqpTOEhZuqSqqWnaKRgv-9ABYdajFUB0WugPGh-SG-...,Eric Tie,https://play-lh.googleusercontent.com/a-/AOh14...,I cannot open the app anymore,1,0,5.4.0.6,2020-10-27 21:24:41,NaN,NaN,newest,com.anydo
1,gp:AOqpTOH0WP4IQKBZ2LrdNmFy_YmpPCVrV3diEU9KGm3...,john alpha,https://play-lh.googleusercontent.com/a-/AOh14...,I have been begging for a refund from this app...,1,0,NaN,2020-10-27 14:03:28,"Please note that from checking our records, yo...",2020-10-27 15:05:52,newest,com.anydo
2,gp:AOqpTOEMCkJB8Iq1p-r9dPwnSYadA5BkPWTf32Z1azu...,Sudhakar .S,https://play-lh.googleusercontent.com/a-/AOh14...,Very costly for the premium version (approx In...,1,0,NaN,2020-10-27 08:18:40,NaN,NaN,newest,com.anydo
3,gp:AOqpTOGFrUWuKGycpje8kszj3uwHN6tU_fd4gLVFy9z...,SKGflorida@bellsouth.net DAVID S,https://play-lh.googleusercontent.com/-75aK0WF...,"Used to keep me organized, but all the 2020 UP...",1,0,NaN,2020-10-26 13:28:07,What do you find troublesome about the update?...,2020-10-26 14:58:29,newest,com.anydo
4,gp:AOqpTOHls7DW8wmDFzTkHwxuqFkdNQtKHmO6Pt9jhZE...,Louann Stoker,https://play-lh.googleusercontent.com/-pBcY_Z-...,Dan Birthday Oct 28,1,0,5.6.0.7,2020-10-26 06:10:50,NaN,NaN,newest,com.anydo


In [5]:
df.shape


(12495, 12)

In [6]:
df['score'].unique()


array([1, 2, 3, 4, 5])

### count of different values for score

In [7]:
df['score'].value_counts()

score
5    2879
4    2775
1    2506
2    2344
3    1991
Name: count, dtype: int64

score 3 has very less count 

### making a new dataframe

In [8]:
df = df[["content", "score"]]

In [9]:
print(df)

                                                 content  score
0                          I cannot open the app anymore      1
1      I have been begging for a refund from this app...      1
2      Very costly for the premium version (approx In...      1
3      Used to keep me organized, but all the 2020 UP...      1
4                                    Dan Birthday Oct 28      1
...                                                  ...    ...
12490  I really like the planner, it helps me achieve...      5
12491                                             😁****😁      5
12492                  Very useful apps. You must try it      5
12493  Would pay for this if there were even more add...      5
12494                                         Sooow good      5

[12495 rows x 2 columns]


In [10]:
df.isnull().sum()

content    0
score      0
dtype: int64

In [12]:
df.shape


(12495, 2)

#### checking duplicate values

In [13]:
dup=df.duplicated()
print(dup.sum())

551


In [14]:
df=df.drop_duplicates()

In [15]:
df.shape

(11944, 2)

#### labeleing the sentiment 


In [16]:
def get_sentiment(score):
    if score >= 4:
        return "Positive"
    elif score == 3:
        return "Neutral"
    else:
        return "Negative"

df["sentiment"] = df["score"].apply(get_sentiment)

In [17]:
df.head(5)

,content,score,sentiment
0,I cannot open the app anymore,1,Negative
1,I have been begging for a refund from this app...,1,Negative
2,Very costly for the premium version (approx In...,1,Negative
3,"Used to keep me organized, but all the 2020 UP...",1,Negative
4,Dan Birthday Oct 28,1,Negative


In [21]:
avg_word_count = df['content'].str.split().str.len().mean()
print(avg_word_count)

28.875334896182185


#### Remove the words with less than 4 length and more than 100 words

In [22]:
df = df[df["content"].str.split().str.len().between(4, 75)]

In [23]:
df.count()

content      9657
score        9657
sentiment    9657
dtype: int64

#### cleaning the content section

In [24]:
import re

def clean_text(text):
    # convert to string and lower
    text = str(text).lower()
    
    # removing urls
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    
    # handle contractions 
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'s", " is", text)
    
    # removing emojis and non-ASCII
    text = text.encode("ascii", "ignore").decode("utf-8")
    
    # removing special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    
    # Removing extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text if len(text) > 0 else "empty_review"

In [25]:
df["reviews"] = df["content"].apply(clean_text)

C:\Users\aryan\AppData\Local\Temp\ipykernel_21280\591729342.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["reviews"] = df["content"].apply(clean_text)


### Applying lemmatization

In [26]:
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('punkt')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [27]:
# This function converts NLTK's POS tags to WordNet's format
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ        # Adjective: "better" → "good"
    elif tag.startswith('V'):
        return wordnet.VERB       # Verb: "crashing" → "crash"
    elif tag.startswith('R'):
        return wordnet.ADV        # Adverb: "badly" → "bad"
    else:
        return wordnet.NOUN       # Default: treat as noun

def apply_lemmatization(text):
    # Step 1: Tokenize into words
    words = word_tokenize(text)

    # Step 2: Get POS tag for each word (e.g., "crashing" → "VBG" meaning verb)
    pos_tags = pos_tag(words)

    # Step 3: Lemmatize each word using its correct POS
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags
    ]

    return " ".join(lemmatized)

# Apply to dataframe
df['reviews'] = df['reviews'].apply(apply_lemmatization)

# Remove empty rows
df = df[df['reviews'].str.strip() != ""]

C:\Users\aryan\AppData\Local\Temp\ipykernel_21280\700008160.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviews'] = df['reviews'].apply(apply_lemmatization)


In [28]:
test_cases = [
    "the app keeps crashing and running slowly",
    "this is better than before",
    "updates are not working properly",
    "i am loving this application"
]

for t in test_cases:
    print(f"Before: {t}")
    print(f"After:  {apply_lemmatization(t)}")
    print()

Before: the app keeps crashing and running slowly
After:  the app keep crash and run slowly

Before: this is better than before
After:  this be good than before

Before: updates are not working properly
After:  update be not work properly

Before: i am loving this application
After:  i be love this application



In [30]:
print(df['reviews'].head(25))

0                        i can not open the app anymore
1     i have be beg for a refund from this app for o...
2     very costly for the premium version approx ind...
3     use to keep me organize but all the update hav...
4                                      dan birthday oct
5     it have change how i view my different list no...
7     reset my free trial new phone i d like to see ...
8     how do to stop monthly payment because i do no...
11    widget be useless because they always show all...
12    horrible app it do not do a described absolute...
14    do not realize there be a charge for this app ...
15    spams notification of how many task other user...
16    download first thing it try to do be do a con ...
17                no longer work with alexa what happen
18                              it do not show any list
20                   this app be not what i want at all
21    unnecessary pop up really annoy and counterpro...
22    they recently start send spammy notificati

In [31]:
avg_word_count = df['reviews'].str.split().str.len().mean()
print(avg_word_count)

25.281557419488454


In [32]:
df['reviews'].head()

0                       i can not open the app anymore
1    i have be beg for a refund from this app for o...
2    very costly for the premium version approx ind...
3    use to keep me organize but all the update hav...
4                                     dan birthday oct
Name: reviews, dtype: object

#### urgency labeling


In [33]:
high_keywords = [
    "crash", "crashes", "crashing",
    "error", "errors","buggy","awful",
    "not working", "doesnt work", "doesn't work","paid","pay",
    "failed", "failure","delete","deleted",
    "cannot", "can't", "cant","cancel","delete","login failed","slow","sign in issue","sign in",
    "refund", "money lost", "charged","disappointing","unusable","annoying","hate", "useless", "waste", "waste of time","waste of money", "ridiculous", "annoying",
    "irritating", "broken", "horrible experience","bad experience", "very bad", "worst app ever",
    "bug", "bugs","money","greedy","frustrated","frustrating","screen stuck","stop","stopped","not responding","blocked"
    "issue", "issues","worst","worse","pathetic","complicated","blank screen","black screen","white screen",
    "login problem", "cant login", "cannot login","responding","force close","force","credited"
    "stuck", "freeze", "freezing", "hang", "hanging","uninstall","timeout", "request failed","sync failed", "not syncing",
    "data lost", "lost data","dissappointed","billing","subscription","unauthorized",
    "not opening", "wont open", "won't open","terrible","scam","fauad","cheated",
    "server down", "network error","uninstalled"
    "payment failed", "transaction failed",
    "account hacked", "security issue","login","hangs","lags","responsive","irresponsive"
]

In [34]:
medium_keywords = [
    "slow", "sluggish",
    "lag", "lagging","dislike","sad","bad",
    "delay", "delayed","usable","disable","cancel","cancelled",
    "problem", "problems","spam","please",
    "glitch", "glitches","expensive","difficult","unresponsive",
    "takes time", "loading issue","poor","hate",
    "performance issue","permium","unfortunately","failed",
    "unresponsive","least","worthless","shame",
    "drains battery", "battery issue","problem","expensive","terrible",
    "heating", "overheating","responding","complex",
    "update issue","working","broken","fix","ads","too many ads","bloatware","overloaded","bad interfaace","interface",
    "not smooth","awful","ok","slow","clutter","useless",
    "sometimes works","complicated","useless",
    "inconsistent","bad","confused","confuse","integration","pofile not loading","retry",
    "minor bug","annoying","annoy","slow","slow loading", "slight lag", "bit slow", "takes longer", "delayed response", "not fast",
    "average performance", "not optimized","mess","messy","poor design","not reliable","missing","missing features","fix","needs fix","needs update",
    "needs improvement" ,"unnecessary","spamming","missing","costly","battery","battery drain","heavy app","network issue","sync issue","profile"
]

In [35]:
low_keywords = [
    "suggest", "suggestion",
    "feature request",
    "add feature", "add option",
    "please add","nice",
    "would be nice","would like", "i would like", "it would help","consider adding", "can you add", "maybe add",
    "looking forward", "in future updates","please consider", "idea", "request",
    "it would be helpful", "small improvement",
    "improve ui", "better design","ui could be better", "ux could improve","layout improvement", "design could improve",
    "make it simpler", "simplify", "more options","more features", "customization", "flexibility"
    "make it better","better","design" "improve","improvement","enhancement","could be improved",
    "optional","i think", "in my opinion", "feels like",
    "nice to have","good", "pretty good", "decent","works fine", "works well", "okay app", "not bad", "satisfactory", "cool",
    "liked it", "love it", "useful","recommend","wish","hope" ,"add",
    "future" "update","thanks","amazing","great"
]

In [36]:
def get_urgency(text):
    text = str(text).lower()
    
    if any(word in text for word in high_keywords):
        return "High"
    elif any(word in text for word in medium_keywords):
        return "Medium"
    elif any(word in text for word in low_keywords):
        return "Low"
    else:
        return "Low"

#### Making a new Urgency Coloum

In [37]:
df["urgency"] = df["reviews"].apply(get_urgency)

In [38]:
final_df = df[["reviews", "sentiment", "urgency"]]

print(final_df.head(20))

                                              reviews sentiment urgency
0                      i can not open the app anymore  Negative     Low
1   i have be beg for a refund from this app for o...  Negative    High
2   very costly for the premium version approx ind...  Negative  Medium
3   use to keep me organize but all the update hav...  Negative    High
4                                    dan birthday oct  Negative     Low
5   it have change how i view my different list no...  Negative    High
7   reset my free trial new phone i d like to see ...  Negative     Low
8   how do to stop monthly payment because i do no...  Negative    High
11  widget be useless because they always show all...  Negative    High
12  horrible app it do not do a described absolute...  Negative     Low
14  do not realize there be a charge for this app ...  Negative     Low
15  spams notification of how many task other user...  Negative    High
16  download first thing it try to do be do a con ...  Negative 

#### Saving the proceesed Dataset

In [39]:
final_df.to_csv("../data/processed/playstore_clean.csv", index=False)